# Knowledge Distillation — Teaching a Tiny Network with a Big Teacher

## BMVA Summer School — Practical Session (Advanced)

<a href="https://colab.research.google.com/github/Adaptive-Intelligence-Lab/BMVA-summer-school/blob/main/BMVA_CVSS_Advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

Welcome back! In the [beginner practical](https://colab.research.google.com/github/Adaptive-Intelligence-Lab/BMVA-summer-school/blob/main/BMVA_CVSS_Beginner.ipynb) you built CNN classifiers and a segmentation
model by training them directly on labelled data. This practical asks a different question: **what if you
barely have any labels — but you do have a big, accurate model?** The answer is **knowledge distillation
(KD)** (Hinton, Vinyals & Dean, 2015): a small "student" network learns from a large "teacher" network's
predictions, and ends up far stronger than it could ever get from the few labels alone. We will see it
produce a **large, reliable accuracy gain** on CIFAR-10 — and we will *see* the difference, not just read
a number.

The **core path** (Parts 0–13) takes you from raw data to a distilled student and the visualisations that
explain *why* it works. Parts 14–15, which compare distillation flavours (hard pseudo-labels, Decoupled KD),
are an **optional extension** if time allows.

### How this notebook works

- Sections marked **✏️ Exercise** contain a skeleton with `# TODO` markers. Try to fill them in yourself first.
- Every exercise is immediately followed by a **✅ Reference solution** cell. The rest of the notebook always
  uses the reference solution, so it still runs top-to-bottom even if you skip an exercise — you can always
  come back and compare your attempt against it. *(For the training-call exercises, the reference cell only
  trains if your attempt didn't — nothing is trained twice.)*
- **🧠 Quick check** boxes ask a short question about what you have just seen — think first, then click to
  reveal the answer.
- Parts 14–15 are an **optional extension** — skip straight to the Wrap-up if you're short on time.

There are four exercises:

| Exercise | What you implement | Where |
|:---|:---|:---|
| 1 | the distillation loss `kd_loss` | Part 4 |
| 2 | the training loop `train` | Part 5 |
| 3 (a / b / c) | the three training calls: teacher, student-alone, distilled student | Parts 6, 9, 10 |
| 4 | the Decoupled-KD loss `dkd_loss` *(optional)* | Part 15 |

### What's inside

**Core:**

| Part | Topic |
|:---|:---|
| 0 | Setup |
| 1 | What is knowledge distillation? |
| 2 | Dataset: CIFAR-10 |
| 3 | The models: a big teacher and a tiny student |
| 4 | The distillation loss |
| 5 | The training loop |
| 6 | Train the teacher |
| 7 | Seeing the "dark knowledge" |
| 8 | The label-scarce setup |
| 9 | Baseline: the student on its own |
| 10 | Distil: the same student, taught by the teacher |
| 11 | See the difference |
| 12 | Where each model looks |
| 13 | Finale: distillation with zero labels |

**Optional extension:**

| Part | Topic |
|:---|:---|
| 14 | Hard pseudo-labels vs soft distillation |
| 15 | Decoupled KD (DKD) |

This notebook needs `torch`, `torchvision`, `matplotlib`, `numpy` and `scikit-learn` — all pre-installed on
Google Colab. A GPU runtime is recommended (`Runtime > Change runtime type > GPU`); everything also runs on
CPU, just slower (the teacher takes ~5 min).

### Prerequisites

- Comfortable with PyTorch `nn.Module`, basic CNNs and the 5-step training loop — covered in the
  [beginner practical](https://colab.research.google.com/github/Adaptive-Intelligence-Lab/BMVA-summer-school/blob/main/BMVA_CVSS_Beginner.ipynb).
- No prior distillation experience required.

---
# Part 0 — Setup

Run this on **Google Colab with a GPU runtime** (`Runtime > Change runtime type > GPU`) if possible — the
whole core path then takes only a few minutes.

One engineering trick makes that speed possible: we keep the **entire dataset in GPU memory as one tensor**
and do **augmentation on the GPU**. This avoids the CPU/disk `DataLoader` bottleneck (Colab machines have
only 2 CPU cores) and is the main reason the notebook trains in minutes rather than hours.

In [ ]:
# Uncomment the line below if you are running this on a fresh local environment
# (everything here is pre-installed on Google Colab).
# !pip install -q torch torchvision matplotlib numpy scikit-learn

import os, time, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision as tv
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print('PyTorch', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cpu':
    print('No GPU found — it still works, but the teacher will take a few minutes. '
          'Use Runtime → Change runtime type → GPU for the fast path.')

# ----- knobs you can experiment with later -----
N_LABELS       = 1000   # ground-truth labels the student is allowed to use
TEMP           = 4.0    # distillation temperature T
ALPHA          = 0.7    # weight of the soft (KD) loss vs the hard (CE) loss
BETA_DKD       = 8.0    # weight of NCKD vs TCKD in Decoupled KD (Part 15)
STUDENT_EPOCHS = 25     # distillation runs in Parts 10 & 13
ADV_EPOCHS     = 12     # Parts 14-15 only — shorter to save ~1 min on Colab; set = STUDENT_EPOCHS for max accuracy
TEACHER_EPOCHS = 18     # ~92%. Lower for speed, higher for accuracy.
SEED           = 0

> **🔧 A note on tooling — what we let PyTorch (and friends) do for us.**
> This notebook is deliberately *practical*: the goal is to **use the standard tools well**, not to re-derive them
> from scratch. Watch for these library building blocks — each one saves you from writing a lot by hand, and we flag
> them with a 🔧 where they first appear:
> - **`torchvision.datasets.CIFAR10`** — downloads & parses the dataset (no manual binary unpacking).
> - **`torch.nn` layers** (`Conv2d`, `BatchNorm2d`, `Linear`, `Sequential`) **+ autograd** — you write only the
>   *forward* pass; PyTorch derives every gradient for `.backward()`. You never hand-code a derivative.
> - **`F.cross_entropy`** — fuses `log_softmax` + negative-log-likelihood into one numerically-stable call.
> - **`F.kl_div` / `F.log_softmax` / `F.softmax`** — the ingredients of the distillation loss (no hand-summed KL).
> - **`torch.optim.SGD` + `OneCycleLR`** — the weight-update rule (momentum, weight decay, Nesterov) and the
>   *one-cycle* learning-rate schedule (warm up to `max_lr`, then anneal towards ~0; it cycles momentum too), so you
>   implement neither.
> - **`sklearn.manifold.TSNE`** — an off-the-shelf 2‑D embedding for visualisation.
>
> The flip side: because these are *library calls*, it is worth knowing **what each one does under the hood** — that
> is the difference between using PyTorch and understanding it.

---
# Part 1 — What is knowledge distillation?

In the beginner practical, every model learned directly from **ground-truth labels**. But real projects often
look like this instead:

- you have a powerful but heavy **teacher** network (too slow / large to deploy),
- you have **lots of images** but only a **handful of labels** (labelling is expensive!),
- and you need a **tiny** model to actually ship.

Training the tiny model on a few labels alone goes badly — it overfits and generalises poorly. **Knowledge
distillation** (Hinton, Vinyals & Dean, 2015) fixes this by letting the teacher act as an *automatic
labeller*: the student trains to match the teacher's **soft predictions** (its full output distribution, not
just the top class) on all the unlabelled images.

Why soft predictions? Because they carry **"dark knowledge"**: when the teacher sees a *cat* it puts a little
probability on *dog* and almost none on *truck*. That relative structure — which classes are confusable with
which — is information a one-hot label throws away, and it turns out to be a remarkably effective teaching
signal. We will visualise it directly in Part 7.

**What you'll build (≈ a few minutes on a Colab T4):**

1. Train a **teacher** CNN (a small ResNet, ~1.08M params).
2. Train a tiny **student** (TinyCNN, ~24k params — **44× smaller**) on only **1000 labels** → weak.
3. **Distil** the *same* student from the teacher → a **large gain (~+15–20 points)**.
4. Visualise *why* it works (dark knowledge, t‑SNE, confusion matrices, attention).
5. Finale: a student that reaches ~78% having seen **zero ground-truth labels**.
6. **Optional:** compare hard pseudo-labels vs soft KD (Part 14), then **Decoupled KD** (Part 15) for a further boost.

![Headline result: t-SNE features, confusion matrices and accuracy bars for the student alone vs the distilled student](https://raw.githubusercontent.com/Adaptive-Intelligence-Lab/BMVA-summer-school/main/assets/kd_viz_preview.png)

*The headline result you will build towards: the* same *24k-parameter student jumps ~19 points when the only
change is learning from the teacher's soft predictions. Every panel of this figure is produced by code you
will run (and partly write) below.*

---
# Part 2 — Dataset: CIFAR-10

We use [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html): 60,000 tiny 32×32 colour images in 10
classes (50,000 train / 10,000 test) — small enough to train on in minutes, hard enough that a tiny network
genuinely struggles, which is exactly the regime where distillation shines.

> **A note on the download.** `torchvision` normally fetches CIFAR-10 from the University of Toronto server,
> which has recently been extremely slow. The cell below therefore first tries a **fast mirror hosted with
> this notebook's GitHub repository**, and only falls back to the official server if that fails. Either way,
> `torchvision` verifies the archive's checksum before unpacking — so you always end up with the genuine
> dataset.

After loading, we normalise with CIFAR-10's per-channel mean/std and move everything to the GPU as one big
tensor (the Part 0 trick).

In [ ]:
import pathlib, urllib.request

# Fast mirror first (the official Toronto server can take hours at the moment), then fallback.
DATA_URL = 'https://github.com/Adaptive-Intelligence-Lab/BMVA-summer-school/releases/download/data-v1.0/cifar-10-python.tar.gz'
archive = pathlib.Path('./data/cifar-10-python.tar.gz')
if not archive.exists():
    archive.parent.mkdir(exist_ok=True)
    try:
        print('Downloading CIFAR-10 from the summer-school mirror (~163 MB)...')
        urllib.request.urlretrieve(DATA_URL, archive)
        print('Done.')
    except Exception as e:
        archive.unlink(missing_ok=True)  # remove any partial file
        print(f'Mirror unavailable ({e}) — torchvision will fall back to the official server.')

MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
STD  = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
CLASSES = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']

def load_split(train):
    # 🔧 torchvision verifies the archive's md5 and unpacks it; it only downloads if no valid archive is present
    d = tv.datasets.CIFAR10('./data', train=train, download=True)
    X = torch.tensor(d.data, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    X = (X - MEAN) / STD                                            # standardise per channel
    return X.to(DEVICE), torch.tensor(d.targets).to(DEVICE)

Xtr, ytr = load_split(True)
Xte, yte = load_split(False)
N = len(Xtr)
print(f'train {N} | test {len(Xte)} | tensors on {Xtr.device}')

### Visualise a few image samples

Always look at your data before writing a single line of model code.

In [ ]:
def unnorm(x):  # undo normalisation for displaying images
    return (x.cpu() * STD + MEAN).clamp(0, 1).permute(0, 2, 3, 1).numpy()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, i in zip(axes.flat, torch.randint(0, N, (16,))):
    ax.imshow(unnorm(Xtr[i:i+1])[0]); ax.set_title(CLASSES[ytr[i]], fontsize=8); ax.axis('off')
plt.suptitle('CIFAR-10 samples'); plt.tight_layout(); plt.show()

---
# Part 3 — The models: a big teacher and a tiny student

- **Teacher** — a small CIFAR ResNet (~1.08M params). Strong, but too heavy to deploy.
- **Student** — `TinyCNN`, three conv blocks (~24k params, **44× smaller**). This is what we actually want to ship.

The student keeps a CNN's useful inductive bias (so it *can* learn well) — it is just **capacity- and label-limited**.

> **🔧 Tooling.** We *compose* both networks from `torch.nn` layers (`Conv2d`, `BatchNorm2d`, `Linear`, `Sequential`)
> and subclass `nn.Module`. Defining the `forward` pass is **all** we do — PyTorch's **autograd** records the
> operations and produces every gradient automatically when we later call `loss.backward()`. (We do write the ResNet
> block by hand here, because seeing the architecture is the point; for real projects you'd often grab one from
> `torchvision.models`.)

The cell also defines `augment`, the GPU-side data augmentation (random flip + random crop) used during
training — vectorised over the whole batch so it costs almost nothing.

In [ ]:
def augment(x, pad=4):
    """GPU-side augmentation: random horizontal flip + reflect-pad random crop (fully vectorised)."""
    B, C, H, W = x.shape
    flip = torch.rand(B, device=x.device) < 0.5
    x = torch.where(flip.view(B, 1, 1, 1), x.flip(-1), x)
    xp = F.pad(x, (pad, pad, pad, pad), mode='reflect')
    oy = torch.randint(0, 2*pad+1, (B,), device=x.device)
    ox = torch.randint(0, 2*pad+1, (B,), device=x.device)
    r = oy[:, None] + torch.arange(H, device=x.device)[None, :]   # [B,H]
    c = ox[:, None] + torch.arange(W, device=x.device)[None, :]   # [B,W]
    b = torch.arange(B, device=x.device)[:, None, None]
    return xp[b, :, r[:, :, None], c[:, None, :]].permute(0, 3, 1, 2).contiguous()


class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);      self.b2 = nn.BatchNorm2d(cout)
        self.sc = nn.Sequential()
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x))); o = self.b2(self.c2(o)); o += self.sc(x); return F.relu(o)

class ResNet(nn.Module):                       # the TEACHER
    def __init__(self, n=3, widths=(32, 64, 128), nc=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, widths[0], 3, 1, 1, bias=False); self.bn1 = nn.BatchNorm2d(widths[0])
        layers, cin = [], widths[0]
        for i, w in enumerate(widths):
            for b in range(n):
                stride = 2 if (b == 0 and i > 0) else 1
                layers.append(BasicBlock(cin, w, stride)); cin = w
        self.layers = nn.Sequential(*layers); self.fc = nn.Linear(widths[-1], nc)
    def fmap(self, x):                          # last conv feature map (for attention viz)
        return self.layers(F.relu(self.bn1(self.conv1(x))))
    def forward(self, x):
        return self.fc(F.adaptive_avg_pool2d(self.fmap(x), 1).flatten(1))

class TinyCNN(nn.Module):                       # the STUDENT
    def __init__(self, c=16, nc=10):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(3, c, 3, 1, 1, bias=False),     nn.BatchNorm2d(c),    nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(c, 2*c, 3, 1, 1, bias=False),   nn.BatchNorm2d(2*c),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(2*c, 4*c, 3, 1, 1, bias=False), nn.BatchNorm2d(4*c),  nn.ReLU())
        self.fc = nn.Linear(4*c, nc)
    def fmap(self, x):                           # last conv feature map (for attention viz)
        return self.f(x)
    def features(self, x):                       # penultimate pooled features (for t-SNE)
        return F.adaptive_avg_pool2d(self.fmap(x), 1).flatten(1)
    def forward(self, x):
        return self.fc(self.features(x))

n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"Teacher (ResNet):  {n_params(ResNet()):,} params")
print(f"Student (TinyCNN): {n_params(TinyCNN()):,} params  ->  "
      f"{n_params(ResNet())/n_params(TinyCNN()):.0f}x smaller")

---
### 🧠 Quick check 1

**Q:** In the beginner practical's `SimpleCNN`, ~97% of all parameters lived in the first fully-connected
layer. `TinyCNN` also ends with a `Linear` layer — yet it holds almost none of the parameters. Where do most
of TinyCNN's ~24k parameters live, and what design choice makes its `Linear` layer so cheap?

<details><summary><b>Show answer</b></summary>

Most of the parameters are in the **third conv layer**: $32 \times 64 \times 3 \times 3 = 18{,}432$ weights —
about 76% of the total. The `Linear` layer is only $64 \times 10 + 10 = 650$ parameters because of
**global average pooling** (`F.adaptive_avg_pool2d(..., 1)`): instead of flattening a whole $64 \times 8
\times 8$ feature map (which would need a $4096 \times 10$ classifier), each of the 64 channels is averaged
down to a single number first. This is the standard modern trick for avoiding the huge fully-connected
layers you saw in the beginner practical.

</details>

---

---
# Part 4 — The distillation loss

This is the part to get right. The teacher's **softened probabilities** carry the dark knowledge, and the
student learns by matching them:

$$\mathcal{L}_{KD}=T^2\cdot \mathrm{KL}\!\Big(\mathrm{softmax}(z_t/T)\;\Big\|\;\mathrm{softmax}(z_s/T)\Big)$$

where $z_t, z_s$ are the teacher's and student's raw logits and $T$ is the **temperature** — dividing the
logits by $T>1$ before the softmax spreads probability onto the smaller classes, exposing the relative
structure that a hard label hides.

*(We minimise $\mathrm{KL}(p_t\Vert p_s)$; it differs from Hinton's original cross-entropy form only by the teacher's
constant entropy, so the student's gradients are identical — and `F.kl_div` is the numerically-stable implementation.)*

Three classic gotchas:

1. **Apply temperature to raw logits** (`softmax(z/T)`), never re-softmax probabilities.
2. **Multiply the soft loss by `T²`.** Soft-target gradient magnitudes scale like `1/T²` (Hinton's small-logit
   approximation), so the `T²` keeps the soft term on the same scale as the hard cross-entropy term — otherwise the
   KD term is silently down-weighted as `T` grows and `ALPHA` stops meaning what you think.
3. **`F.kl_div` wants log-probabilities as its first argument**, and use **`reduction='batchmean'`** — it divides by
   the batch size only (matching the KL definition); the default `'mean'` also divides by the number of classes,
   making the loss ~10× too small here.

> **🔧 Tooling.** For the *hard-label* term (later) we use **`F.cross_entropy`** (which internally does `log_softmax` +
> negative-log-likelihood, fused and numerically stable). For the *soft* term we assemble
> **`F.kl_div(F.log_softmax(...), F.softmax(...))`** rather than summing the KL divergence by hand.

**✏️ Exercise 1:** implement `kd_loss_exercise` below, following the three steps in its docstring.

In [ ]:
def kd_loss_exercise(student_logits, teacher_logits, T):
    """Hinton soft-target loss: KL between the teacher's and student's T-softened output
    distributions, scaled by T**2.  (The formula and the 3 gotchas are in the section above.)
    TODO (in words):
      1. turn the STUDENT logits into log-probabilities at temperature T,
      2. turn the TEACHER logits into probabilities at temperature T,
      3. take the KL divergence with reduction='batchmean', then multiply by T**2  (gotcha #2).
    Useful functions: F.log_softmax, F.softmax, F.kl_div   (mind F.kl_div's argument order — gotcha #3).
    """
    raise NotImplementedError("Exercise 1: implement kd_loss")

### ✅ Reference solution

In [ ]:
def kd_loss(student_logits, teacher_logits, T):
    """Hinton soft-target loss: KL between temperature-softened distributions, scaled by T^2."""
    return F.kl_div(F.log_softmax(student_logits / T, dim=1),   # 🔧 F.kl_div wants LOG-probs first
                    F.softmax(teacher_logits / T, dim=1),
                    reduction='batchmean') * (T * T)            # the T^2 factor (gotcha #2)


# quick unit test: identical logits -> zero loss; disagreeing logits -> positive loss
_z = torch.randn(4, 10)
print(f'kd_loss(z, z)          = {kd_loss(_z, _z, T=TEMP).item():.6f}   (expect 0)')
print(f'kd_loss(z, shuffled z) = {kd_loss(_z, _z[:, torch.randperm(10)], T=TEMP).item():.4f}   (expect > 0)')

---
### 🧠 Quick check 2

**Q:** Suppose you used `reduction='mean'` instead of `'batchmean'` in `F.kl_div`. The notebook would still
run without any error — so what would silently go wrong?

<details><summary><b>Show answer</b></summary>

`'mean'` divides the summed KL by **every element** of the tensor — batch size × number of classes — while the
mathematical definition of the batch-averaged KL divides by the batch size only. With 10 classes the KD loss
would come out **10× too small**, so the soft-target signal would be quietly drowned out by the hard
cross-entropy term, and `ALPHA` would no longer mean what you set it to. Distillation would "work", just far
worse than it should — one of the classic silent bugs in KD implementations.

</details>

---

### Aside — *where* can you distill? (what the student matches)

"Knowledge" can be transferred from more than one place in the teacher. This notebook uses the first (and simplest)
kind; the others are what the "where to go next" pointers at the end refer to.

| What the student matches | Name | What it transfers | Effect / trade-off |
|---|---|---|---|
| **Output logits** (softened) | **response / logit-based** (Hinton 2015) — *what we do* | the teacher's class probabilities — "dark knowledge" about which classes are confusable | simplest, architecture-agnostic, a strong baseline; needs only the teacher's outputs |
| **Intermediate features** | **feature-based** — FitNets (Romero 2015) | *how* the teacher represents the input internally | richer signal, can help when logits aren't enough — but you must choose *which* layer, add a projector to match shapes, and weight it carefully (**too strong can hurt**) |
| **Spatial attention maps** | **attention transfer** (Zagoruyko 2017) — *we visualise this in Part 12* | *where* the teacher looks | a light, channel-count-agnostic form of feature transfer |
| **Relations between samples** | **relational KD** (Park 2019) | the geometry of the feature space (distances/angles between examples), not single points | transfers *structure* rather than pointwise activations |

**Different place → different effect.** Logit-based aligns the *task output*; feature/attention-based align the
*internal representation* (more information, but more knobs and more ways to go wrong); relational aligns *structure*.
Which one wins is **task-dependent** — for our label-scarce transfer-set task, plain logit-based KD is already very
strong, and in our own tests adding feature-based matching did **not** beat it. That design space is a big part of
what makes KD interesting to research.

---
# Part 5 — The training loop

One `train` function will drive *every* experiment in this notebook. Its arguments select the mode:

| Arguments | Mode | Used in |
|:---|:---|:---|
| `teacher_logits=None` | plain cross-entropy (baseline) | Parts 6, 9 |
| `teacher_logits` set, `kd_mode='soft'` | Hinton soft-target KD *(default)* | Parts 10, 13 |
| `teacher_logits` set, `kd_mode='hard_pseudo'` | CE on the teacher's argmax for unlabelled images | Part 14 |
| `teacher_logits` set, `kd_mode='dkd'` | Decoupled KD | Part 15 |

With `labeled=<subset>` it runs in **transfer-set** mode: the soft/DKD loss applies to *all* images, while
the hard cross-entropy applies only to the labelled subset.

First, two small provided helpers: `accuracy` (batched evaluation) and `teacher_soft_labels` (one forward
pass over the whole training set, caching the teacher's logits — after this the big teacher never has to run
again, so students train in seconds).

> **🔧 Tooling.** Inside `train`, **`torch.optim.SGD`** applies the weight update (momentum / weight-decay /
> Nesterov) and **`OneCycleLR`** schedules the learning rate — and the single line `loss.backward()` runs
> autograd over everything.

In [ ]:
@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    preds = torch.cat([model(X[i:i+1000]).argmax(1) for i in range(0, len(X), 1000)])
    return 100 * (preds == y).float().mean().item()


@torch.no_grad()
def teacher_soft_labels(model):
    """One forward pass over all (un-augmented) training images -> cached teacher logits.
    Distillation then becomes 'train on a soft-labelled dataset', and the big teacher is never
    re-run during student training, so students train in seconds."""
    model.eval()
    return torch.cat([model(Xtr[i:i+1000]) for i in range(0, N, 1000)])

**✏️ Exercise 2:** complete `train_exercise` below. The optimiser, LR schedule, label mask and the two
advanced modes (`hard_pseudo`, `dkd` — for the optional extension) are already provided; your job is the
**one soft-KD / baseline training step** marked `TODO` — the same five steps as every training loop in the
beginner practical, plus the mixed loss.

In [ ]:
def train_exercise(model, idx, epochs, *, teacher_logits=None, labeled=None,
                   T=TEMP, alpha=ALPHA, bs=256, lr=0.1, aug=True, kd_mode='soft'):
    """Train `model` on the images in `idx`. Modes (chosen by the arguments):
       - teacher_logits=None                         -> plain cross-entropy (baseline)
       - teacher_logits set, kd_mode='soft'          -> Hinton soft-target KD (default; Parts 10, 13)
       - teacher_logits set, kd_mode='hard_pseudo'   -> CE on teacher argmax for unlabelled (Part 14, PROVIDED)
       - teacher_logits set, kd_mode='dkd'           -> Decoupled KD (Part 15, uses dkd_loss — Exercise 4)
       With `labeled=subset`: transfer-set — soft/DKD on all images, CE only on `labeled`;
       hard_pseudo uses ground-truth CE on labelled + teacher-argmax CE on unlabelled.
    """
    model = model.to(DEVICE).train()
    # -- PROVIDED (the PyTorch tooling from the note above): optimiser + LR schedule + label mask --
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)   # 🔧
    idx = torch.as_tensor(idx, device=DEVICE)
    steps = (len(idx) + bs - 1) // bs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs, steps_per_epoch=steps)   # 🔧
    lab = None
    if labeled is not None:
        lab = torch.zeros(N, dtype=torch.bool, device=DEVICE)
        lab[torch.as_tensor(labeled, device=DEVICE, dtype=torch.long)] = True

    for ep in range(epochs):
        model.train()
        perm = idx[torch.randperm(len(idx), device=DEVICE)]
        for i in range(0, len(perm), bs):
            bi = perm[i:i+bs]
            x = augment(Xtr[bi]) if aug else Xtr[bi]

            # ---- Advanced modes for Parts 14-15 (PROVIDED — leave as-is) ----
            if kd_mode == 'hard_pseudo' and teacher_logits is not None:
                logits = model(x); loss = 0.0
                if lab is not None:
                    m, um = lab[bi], ~lab[bi]
                    if m.any():  loss = loss + F.cross_entropy(logits[m], ytr[bi][m])
                    if um.any(): loss = loss + F.cross_entropy(logits[um], teacher_logits[bi][um].argmax(1))
                else:
                    loss = loss + F.cross_entropy(logits, teacher_logits[bi].argmax(1))
                opt.zero_grad(); loss.backward(); opt.step(); sched.step()
                continue
            if kd_mode == 'dkd' and teacher_logits is not None:
                logits = model(x)
                if lab is not None:
                    target = ytr[bi].clone()
                    target[~lab[bi]] = teacher_logits[bi][~lab[bi]].argmax(1)
                    loss = alpha * dkd_loss(logits, teacher_logits[bi], target, T)
                    m = lab[bi]
                    if m.any(): loss = loss + F.cross_entropy(logits[m], ytr[bi][m])
                else:
                    target = teacher_logits[bi].argmax(1)
                    loss = dkd_loss(logits, teacher_logits[bi], target, T) + (1 - alpha) * F.cross_entropy(logits, ytr[bi])
                opt.zero_grad(); loss.backward(); opt.step(); sched.step()
                continue

            # ================ TODO (Exercise 2): write ONE soft-KD / baseline training step ================
            # (1) forward:  logits = model(x)
            # (2) build `loss` for the current mode:
            #        loss = 0.0
            #        soft term — only if teacher_logits is not None:
            #             kd = kd_loss(logits, teacher_logits[bi], T)
            #             transfer-set mode (lab is not None):  loss += alpha * kd
            #             soft+hard mode    (lab is None):      loss += kd
            #        hard term (cross-entropy):
            #             lab is None:  ce = F.cross_entropy(logits, ytr[bi])
            #                           loss += (1 - alpha) * ce   (or just ce if teacher_logits is None)
            #             lab is set :  m = lab[bi]; if m.any(): loss += F.cross_entropy(logits[m], ytr[bi][m])
            # (3) update:   opt.zero_grad(); loss.backward(); opt.step()
            # (4) schedule: sched.step()
            raise NotImplementedError("Exercise 2: implement one training step")
            # ===========================================================================
    return model

### ✅ Reference solution

In [ ]:
def train(model, idx, epochs, *, teacher_logits=None, labeled=None,
          T=TEMP, alpha=ALPHA, bs=256, lr=0.1, aug=True, kd_mode='soft'):
    """Train `model` on the images in `idx`. Modes (chosen by the arguments):
       - teacher_logits=None                         -> plain cross-entropy (baseline)
       - teacher_logits set, kd_mode='soft'          -> Hinton soft-target KD (default; Parts 10, 13)
       - teacher_logits set, kd_mode='hard_pseudo'   -> CE on teacher argmax for unlabelled (Part 14)
       - teacher_logits set, kd_mode='dkd'           -> Decoupled KD (Part 15, uses dkd_loss — Exercise 4)
       With `labeled=subset`: transfer-set — soft/DKD on all images, CE only on `labeled`;
       hard_pseudo uses ground-truth CE on labelled + teacher-argmax CE on unlabelled.
    """
    model = model.to(DEVICE).train()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)   # 🔧
    idx = torch.as_tensor(idx, device=DEVICE)
    steps = (len(idx) + bs - 1) // bs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs, steps_per_epoch=steps)   # 🔧
    lab = None
    if labeled is not None:
        lab = torch.zeros(N, dtype=torch.bool, device=DEVICE)
        lab[torch.as_tensor(labeled, device=DEVICE, dtype=torch.long)] = True

    for ep in range(epochs):
        model.train()
        perm = idx[torch.randperm(len(idx), device=DEVICE)]
        for i in range(0, len(perm), bs):
            bi = perm[i:i+bs]
            x = augment(Xtr[bi]) if aug else Xtr[bi]

            # ---- Advanced modes for Parts 14-15 ----
            if kd_mode == 'hard_pseudo' and teacher_logits is not None:
                logits = model(x); loss = 0.0
                if lab is not None:
                    m, um = lab[bi], ~lab[bi]
                    if m.any():  loss = loss + F.cross_entropy(logits[m], ytr[bi][m])
                    if um.any(): loss = loss + F.cross_entropy(logits[um], teacher_logits[bi][um].argmax(1))
                else:
                    loss = loss + F.cross_entropy(logits, teacher_logits[bi].argmax(1))
                opt.zero_grad(); loss.backward(); opt.step(); sched.step()
                continue
            if kd_mode == 'dkd' and teacher_logits is not None:
                logits = model(x)
                if lab is not None:
                    target = ytr[bi].clone()
                    target[~lab[bi]] = teacher_logits[bi][~lab[bi]].argmax(1)
                    loss = alpha * dkd_loss(logits, teacher_logits[bi], target, T)
                    m = lab[bi]
                    if m.any(): loss = loss + F.cross_entropy(logits[m], ytr[bi][m])
                else:
                    target = teacher_logits[bi].argmax(1)
                    loss = dkd_loss(logits, teacher_logits[bi], target, T) + (1 - alpha) * F.cross_entropy(logits, ytr[bi])
                opt.zero_grad(); loss.backward(); opt.step(); sched.step()
                continue

            # ---- the soft-KD / baseline step (Exercise 2) ----
            logits = model(x)                                            # (1) forward
            loss = 0.0                                                   # (2) build the loss
            if teacher_logits is not None:
                kd = kd_loss(logits, teacher_logits[bi], T)
                loss = loss + (alpha * kd if lab is not None else kd)
            if lab is None:
                ce = F.cross_entropy(logits, ytr[bi])
                loss = loss + ((1 - alpha) * ce if teacher_logits is not None else ce)
            else:
                m = lab[bi]
                if m.any():
                    loss = loss + F.cross_entropy(logits[m], ytr[bi][m])
            opt.zero_grad(); loss.backward(); opt.step()                 # (3) update  🔧 autograd + optimiser
            sched.step()                                                 # (4) LR schedule
    return model

---
# Part 6 — Train the teacher

Standard supervised training on the full labelled set — exactly what you did throughout the beginner
practical, just wrapped in our `train` function. On a Colab T4 this takes ~2–4 minutes; re-running the
notebook reuses the saved `teacher.pt` (delete that file if you change `TEACHER_EPOCHS`).

**✏️ Exercise 3a:** in the cell below, replace the `raise NotImplementedError(...)` line with the right
`train(...)` call. Think: which mode of `train()` gives plain cross-entropy (no distillation)? Which images
does the teacher get to see, and for how many epochs?

In [ ]:
torch.manual_seed(SEED)
teacher = ResNet().to(DEVICE)
ex3a_done = False
if os.path.exists('teacher.pt'):
    print('Found cached teacher.pt — nothing to train here; the reference cell below will load it.')
else:
    try:
        t0 = time.time()
        # TODO (Exercise 3a): train the teacher on ALL images with plain supervised cross-entropy.
        # train(...)
        raise NotImplementedError("Exercise 3a: replace this line with your train(...) call")
        torch.save(teacher.state_dict(), 'teacher.pt')
        print(f'Trained teacher in {time.time()-t0:.0f}s')
        ex3a_done = True
    except NotImplementedError as todo:
        print(f'⚠️ Not finished yet — {todo}')

### ✅ Reference solution

*(Runs only if your exercise cell above didn't already train the teacher. Pressed for time, or on CPU?
Uncomment the two download lines to fetch our pretrained teacher instead of training one.)*

In [ ]:
# Short on time or on CPU? Uncomment these two lines to download the pretrained teacher (~4 MB):
# import urllib.request
# urllib.request.urlretrieve('https://github.com/Adaptive-Intelligence-Lab/BMVA-summer-school/releases/download/data-v1.0/teacher.pt', 'teacher.pt')

if not ex3a_done:
    if os.path.exists('teacher.pt'):
        teacher.load_state_dict(torch.load('teacher.pt', map_location=DEVICE))
        print('Loaded cached teacher.pt')
    else:
        torch.manual_seed(SEED); t0 = time.time()
        train(teacher, range(N), TEACHER_EPOCHS, aug=True)     # plain cross-entropy on all 50k images
        torch.save(teacher.state_dict(), 'teacher.pt')
        print(f'Trained teacher in {time.time()-t0:.0f}s')

print(f'Teacher test accuracy: {accuracy(teacher, Xte, yte):.2f}%')
TL = teacher_soft_labels(teacher).detach()                 # cached soft labels for all 50k images
print('Cached teacher soft labels:', tuple(TL.shape))

---
# Part 7 — Seeing the "dark knowledge"

Two views of what the teacher knows beyond the hard label:
- **Temperature** softens the teacher's prediction on one image, revealing which *other* classes it finds plausible.
- The **per-class soft-target matrix** shows the inter-class structure across the whole dataset — its bright
  off-diagonal entries (e.g. cat↔dog, car↔truck) are exactly the signal hard one-hot labels destroy.

In [ ]:
# (a) temperature on a single ambiguous image
i = (yte == 3).nonzero().flatten()[5].item()       # a 'cat'
with torch.no_grad():
    lg = teacher(Xte[i:i+1])[0]
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
axes[0].imshow(unnorm(Xte[i:i+1])[0]); axes[0].set_title(f'input: {CLASSES[yte[i]]}'); axes[0].axis('off')
for ax, T in zip(axes[1:], [1, 4, 16]):
    p = F.softmax(lg / T, 0).cpu().numpy()
    ax.bar(range(10), p, color='steelblue'); ax.set_ylim(0, 1)
    ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=90, fontsize=8)
    ax.set_title(f'teacher softmax (T={T})')
plt.suptitle('Raising the temperature exposes the teacher\'s dark knowledge'); plt.tight_layout(); plt.show()

# (b) average soft targets per true class
with torch.no_grad():
    soft = F.softmax(TL / TEMP, 1)
M = torch.stack([soft[ytr == c].mean(0) for c in range(10)])
plt.figure(figsize=(5.6, 5))
plt.imshow(M.cpu(), cmap='viridis'); plt.colorbar(fraction=0.046)
plt.xticks(range(10), CLASSES, rotation=90); plt.yticks(range(10), CLASSES)
plt.xlabel('teacher soft probability'); plt.ylabel('true class')
plt.title('Avg teacher soft-targets per class\n(off-diagonal = structure hard labels throw away)')
plt.tight_layout(); plt.show()

---
### 🧠 Quick check 3

**Q:** In the temperature plot, T=1 looks almost one-hot and T=16 looks much flatter. What happens in the
limit $T \to \infty$ — and why is a *moderate* temperature (like our T=4) the sweet spot for distillation?

<details><summary><b>Show answer</b></summary>

As $T \to \infty$, $z/T \to 0$ for every logit, so the softmax tends to the **uniform distribution** — all
classes get probability 0.1 and the teacher's knowledge is erased entirely. At the other extreme, T=1 keeps
the distribution so sharp that it is nearly a one-hot label, hiding the inter-class structure. A moderate T
softens the distribution enough to *expose* the relative logit gaps (cat vs dog vs truck) while still
clearly preferring the right class — that trade-off is why T is a hyperparameter worth sweeping (see the
Wrap-up's "Try it yourself").

</details>

---

---
# Part 8 — The label-scarce setup

The student may use ground-truth labels for only **`N_LABELS`** images (here 1000 = 2%). It can still *see* all
50,000 images — it just has no labels for most of them. This is the realistic regime where labelling is the bottleneck,
and (as we'll see) exactly where distillation pays off the most.

In [ ]:
g = torch.Generator().manual_seed(7)
labeled_idx = torch.randperm(N, generator=g)[:N_LABELS].tolist()
print(f'Student may use labels for {N_LABELS}/{N} images ({100*N_LABELS/N:.0f}%); '
      f'all {N} images are available for distillation.')

---
# Part 9 — Baseline: the student on its own

Train the tiny student on the 1000 labelled images with ordinary cross-entropy. With so few labels it overfits and
generalises poorly.

**✏️ Exercise 3b:** replace the `raise NotImplementedError(...)` line with the right `train(...)` call.
Which index list holds the labelled examples? Use ~60 epochs — and no `teacher_logits` here.

In [ ]:
torch.manual_seed(SEED)
student_alone = TinyCNN()
ex3b_done = False
try:
    t0 = time.time()
    # TODO (Exercise 3b): train `student_alone` on the labelled subset ONLY, with plain cross-entropy.
    # train(...)
    raise NotImplementedError("Exercise 3b: replace this line with your train(...) call")
    ex3b_done = True
except NotImplementedError as todo:
    print(f'⚠️ Not finished yet — {todo}')

### ✅ Reference solution *(trains only if your cell above didn't)*

In [ ]:
if not ex3b_done:
    torch.manual_seed(SEED)
    student_alone = TinyCNN()
    t0 = time.time()
    train(student_alone, labeled_idx, 60, aug=True)      # CE on the 1000 labelled images only

acc_alone = accuracy(student_alone, Xte, yte)
print(f'Student ALONE ({N_LABELS} labels): {acc_alone:.2f}%   ({time.time()-t0:.0f}s)')

---
# Part 10 — Distil: the *same* student, taught by the teacher

Same architecture, same 1000 labels — but now the student also matches the teacher's soft predictions on **all
50,000 images**. In effect the teacher labels the unlabelled data for the student.

**✏️ Exercise 3c:** replace the `raise NotImplementedError(...)` line with the right `train(...)` call.
Which two arguments of `train()` switch on (i) distillation and (ii) "cross-entropy only on the labelled
subset" (the transfer-set mode)? Use `STUDENT_EPOCHS` epochs, on **all** images.

In [ ]:
torch.manual_seed(SEED)
student_kd = TinyCNN()
ex3c_done = False
try:
    t0 = time.time()
    # TODO (Exercise 3c): train `student_kd` on ALL images, distilling from the teacher, with hard labels
    # available ONLY for the labelled subset.
    # train(...)
    raise NotImplementedError("Exercise 3c: replace this line with your train(...) call")
    ex3c_done = True
except NotImplementedError as todo:
    print(f'⚠️ Not finished yet — {todo}')

### ✅ Reference solution *(trains only if your cell above didn't)*

In [ ]:
if not ex3c_done:
    torch.manual_seed(SEED)
    student_kd = TinyCNN()
    t0 = time.time()
    train(student_kd, range(N), STUDENT_EPOCHS, teacher_logits=TL, labeled=labeled_idx)   # KD on all 50k + CE on 1000

acc_kd = accuracy(student_kd, Xte, yte)
print(f'Student DISTILLED (transfer set): {acc_kd:.2f}%   ({time.time()-t0:.0f}s)')
print(f'\n>>> Distillation gain: {acc_kd-acc_alone:+.1f} percentage points — '
      f'same {n_params(TinyCNN()):,}-param student, only the teacher added.')

---
# Part 11 — See the difference

Numbers only tell half the story — the gap between the two students is obvious to the eye:
- **t‑SNE of the student's penultimate features** — the alone-student's classes are smeared together; the distilled
  student's classes form tight, well-separated clusters (more like the teacher's representation).
- **Confusion matrices** — the diagonal gets visibly brighter after distillation.
- **Accuracy bars** — alone → distilled → teacher.

In [ ]:
from sklearn.manifold import TSNE       # 🔧 off-the-shelf 2-D embedding for visualisation

@torch.no_grad()
def feats_preds(model):
    model.eval(); Fs, Ps = [], []
    for i in range(0, len(Xte), 1000):
        f = model.features(Xte[i:i+1000]); Fs.append(f.cpu()); Ps.append(model.fc(f).argmax(1).cpu())
    return torch.cat(Fs).numpy(), torch.cat(Ps).numpy()

Fa, Pa = feats_preds(student_alone)
Fd, Pd = feats_preds(student_kd)
Y = yte.cpu().numpy()
sub = np.random.default_rng(0).choice(len(Y), 2000, replace=False)
print('running t-SNE (~1 min)...')
Za = TSNE(2, init='pca', perplexity=30, random_state=0).fit_transform(Fa[sub])
Zd = TSNE(2, init='pca', perplexity=30, random_state=0).fit_transform(Fd[sub])

def confmat(P):
    Mx = np.zeros((10, 10));
    for p, t in zip(P, Y): Mx[t, p] += 1
    return Mx / Mx.sum(1, keepdims=True)

cmap = plt.get_cmap('tab10'); Ys = Y[sub]
fig = plt.figure(figsize=(15, 9))
def scat(ax, Z, title):
    for c in range(10):
        m = Ys == c; ax.scatter(Z[m, 0], Z[m, 1], s=6, color=cmap(c), label=CLASSES[c], alpha=0.7)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
ax = fig.add_subplot(2, 3, 1); scat(ax, Za, f'Student ALONE — t-SNE ({acc_alone:.0f}%)')
ax = fig.add_subplot(2, 3, 2); scat(ax, Zd, f'Student DISTILLED — t-SNE ({acc_kd:.0f}%)')
ax.legend(markerscale=2, fontsize=7, loc='center left', bbox_to_anchor=(1.0, 0.5))
def cm(ax, Mx, title):
    ax.imshow(Mx, cmap='viridis', vmin=0, vmax=1); ax.set_title(title)
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(CLASSES, rotation=90, fontsize=7); ax.set_yticklabels(CLASSES, fontsize=7)
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax = fig.add_subplot(2, 3, 4); cm(ax, confmat(Pa), f'Confusion — ALONE ({acc_alone:.0f}%)')
ax = fig.add_subplot(2, 3, 5); cm(ax, confmat(Pd), f'Confusion — DISTILLED ({acc_kd:.0f}%)')
ax = fig.add_subplot(2, 3, (3, 6))
vals = [acc_alone, acc_kd, accuracy(teacher, Xte, yte)]
bars = ax.bar(['alone\n(1k labels)', 'distilled\n(transfer)', 'teacher\n(full)'],
              vals, color=['#d62728', '#2ca02c', '#1f77b4'])
ax.set_ylim(0, 100); ax.set_ylabel('test accuracy (%)'); ax.set_title('Accuracy')
for r, v in zip(bars, vals): ax.text(r.get_x()+r.get_width()/2, v+1.5, f'{v:.1f}', ha='center')
ax.annotate(f'+{acc_kd-acc_alone:.0f} pts', xy=(1, acc_kd), xytext=(0.5, acc_kd+12),
            color='green', fontsize=13, ha='center', arrowprops=dict(arrowstyle='->', color='green'))
plt.suptitle('Knowledge distillation on CIFAR-10: same 24k-param student, only the teacher added', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()

#### Which mistakes did distillation fix?
Images the lone student got **wrong** but the distilled student got **right**.

In [ ]:
flip = np.where((Pa != Y) & (Pd == Y))[0][:16]
fig, axes = plt.subplots(2, 8, figsize=(12, 4.6), constrained_layout=True)
for ax, j in zip(axes.flat, flip):
    ax.imshow(unnorm(Xte[j:j+1])[0]); ax.axis('off')
    ax.set_title(f'{CLASSES[Pa[j]]}→{CLASSES[Pd[j]]}\n({CLASSES[Y[j]]})', fontsize=7)
fig.suptitle('alone-prediction → distilled-prediction   (true label)'); plt.show()

---
# Part 12 — Where each model looks

Beyond the feature clusters above, we can ask *where in the image* each model focuses. Summing the squared
activations of the last convolutional feature map over its channels gives a spatial **attention map** — exactly the
quantity used by *attention-transfer* distillation (the "attention" row in the Part 4 table).

- **Quantitatively** (bar, averaged over all 10,000 test images): the lone student's attention is diffuse;
  distillation makes it markedly more **concentrated**, reaching the teacher's level of focus.
- **Qualitatively** (the grid): on selected clear cases where the lone student was *wrong* and the distilled student
  *right*, the lone student's attention is scattered onto the background while the distilled student's snaps onto the
  object — the same focus the teacher shows.

> Attention maps here are only 8×8 (upsampled) and are noisy per-image, so read the **trend**, not any single map —
> the bar chart is the reliable summary, and the grid below shows hand-picked clear examples for teaching.

The two figures below show the **expected output** of the cells that follow (from a reference run of this
notebook) — a handy check that your Exercise 2 and 3 implementations trained correctly:

![Expected output: attention concentration bar chart — student alone is diffuse, distilled reaches the teacher's focus](https://raw.githubusercontent.com/Adaptive-Intelligence-Lab/BMVA-summer-school/main/assets/attn_focus_bar_preview.png)

![Expected output: curated attention grid — the lone student looks at the background, the distilled student snaps onto the object](https://raw.githubusercontent.com/Adaptive-Intelligence-Lab/BMVA-summer-school/main/assets/attn_curated_preview.png)

In [ ]:
# spatial attention = channel-wise sum of squared activations of the last conv feature map
@torch.no_grad()
def attention_map(model, x):
    a = model.fmap(x).pow(2).mean(1)                       # [B, h, w]
    return a / a.amax(dim=(1, 2), keepdim=True).clamp_min(1e-8)

@torch.no_grad()
def concentration(model, X, bs=1000):
    """Per-image focus = 1 - normalised entropy of the attention map (higher = more concentrated)."""
    vals = []
    for i in range(0, len(X), bs):
        a = model.fmap(X[i:i+bs]).pow(2).mean(1).flatten(1)
        p = a / a.sum(1, keepdim=True).clamp_min(1e-8)
        ent = -(p * (p + 1e-12).log()).sum(1)
        vals.append((1 - ent / np.log(p.shape[1])).cpu())
    return torch.cat(vals).numpy()

conc_alone = concentration(student_alone, Xte)
conc_kd    = concentration(student_kd, Xte)
conc_teach = concentration(teacher, Xte)
means = [conc_alone.mean(), conc_kd.mean(), conc_teach.mean()]

plt.figure(figsize=(4.4, 4))
bars = plt.bar(['student\nalone', 'distilled', 'teacher'], means, color=['#d62728', '#2ca02c', '#1f77b4'])
plt.ylabel('attention concentration (higher = more focused)')
plt.title('Distillation focuses the student\n(mean over 10,000 test images)')
for r, v in zip(bars, means): plt.text(r.get_x()+r.get_width()/2, v+0.001, f'{v:.3f}', ha='center')
plt.tight_layout(); plt.show()
print(f'Distilled attention is {means[1]/means[0]-1:+.0%} more concentrated than training alone '
      f'(alone {means[0]:.3f} -> distilled {means[1]:.3f}, teacher {means[2]:.3f}).')

In [ ]:
# curated grid: alone WRONG, distilled RIGHT, ranked by how much MORE focused the distilled student is
@torch.no_grad()
def att_upsampled(model, x):
    a = attention_map(model, x)
    return F.interpolate(a[:, None], size=(32, 32), mode='bilinear', align_corners=False)[0, 0].cpu().numpy()

cand = np.where((Pa != Y) & (Pd == Y))[0]
sel  = cand[np.argsort(-(conc_kd[cand] - conc_alone[cand]))[:6]]
panels = [('teacher', teacher, None), ('student alone', student_alone, Pa), ('distilled', student_kd, Pd)]

fig, axes = plt.subplots(6, 4, figsize=(9, 13), constrained_layout=True)
for r, j in enumerate(sel):
    x = Xte[j:j+1]; img = unnorm(x)[0]
    axes[r, 0].imshow(img); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    axes[r, 0].set_ylabel(f'true:\n{CLASSES[Y[j]]}', rotation=0, labelpad=22, va='center', fontsize=9)
    if r == 0: axes[r, 0].set_title('input')
    for c, (nm, m, P) in enumerate(panels, 1):
        axes[r, c].imshow(img); axes[r, c].imshow(att_upsampled(m, x), cmap='jet', alpha=0.5)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0: axes[r, c].set_title(nm)
        if P is not None:
            ok = P[j] == Y[j]
            axes[r, c].set_xlabel(('OK ' if ok else 'X ') + CLASSES[P[j]],
                                  color=('green' if ok else 'red'), fontsize=9)
fig.suptitle('Where each model looks (last-conv attention) — selected clear examples', fontsize=13)
plt.show()

**…but the student is still not the teacher.** Even though the distilled student now *looks* at images like the
teacher and gained ~15–20 points, it remains well behind the teacher (~70% vs ~92%). Distillation transfers the
teacher's knowledge and attention, but the **24k-param student is fundamentally capacity-limited** — a network 44×
smaller can only absorb so much. **KD dramatically narrows the teacher–student gap; it does not erase it.** Picking
the student size is a deploy-time accuracy-vs-cost trade-off, and distillation makes the cheap (small-model) end of
that trade-off far stronger than training the small model alone.

---
# Part 13 — Finale: distillation with **zero** labels

Drop the ground-truth labels entirely: train the student **only** on the teacher's soft predictions over the 50,000
images (no `CE` term at all). The student never sees a single true label, yet learns to classify CIFAR-10 well —
distillation transferred the teacher's knowledge directly.

In [ ]:
torch.manual_seed(SEED)
student_zero = TinyCNN()
train(student_zero, range(N), STUDENT_EPOCHS, teacher_logits=TL, labeled=[])   # KD only, NO ground-truth labels
acc_zero = accuracy(student_zero, Xte, yte)
print(f'Student with ZERO ground-truth labels: {acc_zero:.2f}%')
print(f'(teacher {accuracy(teacher, Xte, yte):.1f}%  |  student-alone with {N_LABELS} labels {acc_alone:.1f}%)')

---
### 🧠 Quick check 4

**Q:** The zero-label student (~78%) beats the student trained on 1000 *genuine* labels (~52%) by a wide
margin. It never saw a single true label — so where did all that information come from?

<details><summary><b>Show answer</b></summary>

From the **teacher's soft predictions on all 50,000 images**. Each soft target is a slightly noisy but
information-rich label — it encodes not just the most likely class but the whole plausibility ranking — and
there are 50× more of them than the 1000 ground-truth labels. In effect the teacher converted its own
knowledge (learned from the full labelled set) into a *new, automatically-labelled dataset* for the student.
The catch, of course, is that someone still had to label the data that trained the teacher — distillation
moves labels around; it doesn't create information from nothing.

</details>

---

---
# Part 14 (optional extension) — Hard pseudo-labels vs soft distillation

*Short on time? This is a good place to stop — skip to the [Wrap-up](#Wrap-up) at the end. Everything from
here on compares different ways of using the teacher's predictions.*

The teacher can *automatically label* unlabelled images — but **how** should the student use those labels?

- **Hard pseudo-label:** take `argmax(teacher)` and train with ordinary cross-entropy. Simple, but throws away
  the relative probabilities between the other classes (the dark knowledge from Part 7).
- **Soft distillation (Part 10):** match the teacher's full softened distribution with KL divergence.

Same student, same 1000 ground-truth labels, same 50,000-image transfer set — only the supervision signal changes.

**First** (instant, no training): inspect teacher pseudo-label quality. The training comparison happens in
Part 15, together with DKD.

In [ ]:
# Instant — teacher pseudo-label quality on UNLABELLED train images (no extra training)
unlab = torch.ones(N, dtype=torch.bool, device=DEVICE); unlab[labeled_idx] = False
with torch.no_grad():
    probs = F.softmax(TL / TEMP, 1)
    conf, pred = probs.max(1)
    correct = pred == ytr
    cu, cw = conf[unlab & correct].cpu().numpy(), conf[unlab & ~correct].cpu().numpy()
    teach_acc_unlab = 100 * correct[unlab].float().mean().item()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(cu, bins=30, alpha=0.75, label=f'correct ({len(cu):,})', color='#2ca02c', density=True)
axes[0].hist(cw, bins=30, alpha=0.75, label=f'wrong ({len(cw):,})', color='#d62728', density=True)
axes[0].set_xlabel('teacher confidence (max soft prob)'); axes[0].set_ylabel('density')
axes[0].set_title(f'Unlabelled pseudo-labels: {teach_acc_unlab:.1f}% correct')
axes[0].legend(fontsize=8)

# soft target on one cat — hard label throws away the rest
j = (ytr == 3).nonzero().flatten()[12].item()   # cat in TRAIN set (TL is indexed by train id)
p = F.softmax(TL[j] / TEMP, 0).cpu().numpy(); hard = p.argmax()
axes[1].bar(range(10), p, color='steelblue'); axes[1].set_ylim(0, 1)
axes[1].set_xticks(range(10)); axes[1].set_xticklabels(CLASSES, rotation=90, fontsize=8)
axes[1].axvline(hard, color='red', ls='--', lw=1.5)
axes[1].set_title(f'Soft target example\ntrue={CLASSES[ytr[j]]}, hard={CLASSES[hard]}')
plt.suptitle('Hard pseudo-labels keep one class; soft KD keeps the whole distribution', fontsize=11)
plt.tight_layout(); plt.show()
print(f'Wrong pseudo-labels: {len(cw):,}  |  mean conf when wrong: {cw.mean():.2f}  |  when correct: {cu.mean():.2f}')

---
# Part 15 (optional extension) — Decoupled KD (DKD)

**DKD** (Zhao et al., CVPR 2022) splits the same teacher logits into **TCKD** (target-class confidence) and
**NCKD** (dark-knowledge structure among the other classes), and weights them separately. Same `TL` cache,
often **+1–3 pt** over vanilla soft KD.

**✏️ Exercise 4:** implement `dkd_loss_exercise` below, following the four steps in its docstring. The
`train(..., kd_mode='dkd')` path (already provided in Part 5) calls the reference `dkd_loss`; once your
version passes the comparison, you can swap it in.

In [ ]:
def dkd_loss_exercise(student_logits, teacher_logits, target, T, alpha=1.0, beta=BETA_DKD):
    """Decoupled KD (Zhao et al., CVPR 2022): split soft-target matching into
    TCKD (target-class vs the rest) + NCKD (relative structure among the non-target classes).
    TODO (in words):
      1. build gt_mask = one-hot(target) and other_mask = 1 - gt_mask.
      2. TCKD — collapse each T-softened distribution into 2 buckets [p_target, p_non_target]
         (sum the probs under each mask), then KL(teacher_bin || student_bin) * T**2.
         (clamp the student's 2-bucket probs to >=1e-7 before the log for stability.)
      3. NCKD — mask OUT the target class (subtract e.g. 1000 from its logit), then
         F.kl_div(F.log_softmax(student_nt / T), F.softmax(teacher_nt / T), 'batchmean') * T**2.
      4. return alpha * tckd + beta * nckd.
    Useful functions: F.one_hot, F.softmax, F.log_softmax, F.kl_div.
    """
    raise NotImplementedError("Exercise 4: implement dkd_loss")

### ✅ Reference solution

In [ ]:
def dkd_loss(student_logits, teacher_logits, target, T, alpha=1.0, beta=BETA_DKD):
    """Decoupled KD (Zhao et al., CVPR 2022): split soft-target matching into
    TCKD (target-class vs rest) + NCKD (relative structure among non-target classes)."""
    gt_mask = F.one_hot(target, num_classes=student_logits.size(1)).float()
    other_mask = 1.0 - gt_mask
    pred_s = F.softmax(student_logits / T, dim=1)
    pred_t = F.softmax(teacher_logits / T, dim=1)
    # TCKD — binary distillation on [p_target, p_non-target]
    p_s_bin = torch.cat([(pred_s * gt_mask).sum(1, keepdim=True),
                         (pred_s * other_mask).sum(1, keepdim=True)], dim=1).clamp(1e-7, 1.0)
    p_t_bin = torch.cat([(pred_t * gt_mask).sum(1, keepdim=True),
                         (pred_t * other_mask).sum(1, keepdim=True)], dim=1)
    tckd = (p_t_bin * (p_t_bin / p_s_bin).log()).sum(1).mean() * (T * T)
    # NCKD — KL on non-target logits (mask the target class, then softmax)
    logits_s_nt = student_logits - gt_mask * 1000.0
    logits_t_nt = teacher_logits - gt_mask * 1000.0
    nckd = F.kl_div(F.log_softmax(logits_s_nt / T, dim=1),
                    F.softmax(logits_t_nt / T, dim=1),
                    reduction='batchmean') * (T * T)
    return alpha * tckd + beta * nckd

Now train the two remaining students — **hard pseudo-labels** (from Part 14) and **DKD** — on exactly the
same data as before (`ADV_EPOCHS = 12` each, ~30 s on GPU), then compare all four ways of using the teacher.

In [ ]:
# One cell, two modes — hard pseudo then DKD (ADV_EPOCHS each)
torch.manual_seed(SEED)
student_hard = TinyCNN()
t0 = time.time()
train(student_hard, range(N), ADV_EPOCHS, teacher_logits=TL, labeled=labeled_idx, kd_mode='hard_pseudo')
acc_hard = accuracy(student_hard, Xte, yte)
t_hard = time.time() - t0

torch.manual_seed(SEED)
student_dkd = TinyCNN()
t0 = time.time()
train(student_dkd, range(N), ADV_EPOCHS, teacher_logits=TL, labeled=labeled_idx, kd_mode='dkd')
acc_dkd = accuracy(student_dkd, Xte, yte)
t_dkd = time.time() - t0

print(f'HARD pseudo ({ADV_EPOCHS} ep): {acc_hard:.2f}%   ({t_hard:.0f}s)')
print(f'DKD         ({ADV_EPOCHS} ep): {acc_dkd:.2f}%   ({t_dkd:.0f}s)')
print(f'Soft KD (Part 10, {STUDENT_EPOCHS} ep): {acc_kd:.2f}%  |  alone (Part 9): {acc_alone:.2f}%')
print(f'hard vs alone: {acc_hard-acc_alone:+.1f}  |  soft vs hard: {acc_kd-acc_hard:+.1f}  |  DKD vs soft: {acc_dkd-acc_kd:+.1f} pts')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
names = ['alone\n(1k labels)', 'hard pseudo', 'soft KD', 'DKD']
vals  = [acc_alone, acc_hard, acc_kd, acc_dkd]
bars = ax.bar(names, vals, color=['#d62728', '#ff7f0e', '#2ca02c', '#9467bd'])
ax.set_ylim(0, 100); ax.set_ylabel('test accuracy (%)')
ax.set_title('Same 24k-param student — four ways to use the teacher')
for r, v in zip(bars, vals): ax.text(r.get_x()+r.get_width()/2, v+1.2, f'{v:.1f}', ha='center')
ax.annotate(f'+{acc_dkd-acc_alone:.0f} pts', xy=(3, acc_dkd), xytext=(2.2, acc_dkd+10),
            color='#9467bd', ha='center', fontsize=11, arrowprops=dict(arrowstyle='->', color='#9467bd'))
plt.tight_layout(); plt.show()

---
# Wrap-up

Well done! In this practical you covered the complete recipe for **knowledge distillation**:

| Ingredient | What we used |
|:---|:---|
| **Data** | CIFAR-10 via `torchvision` (fast mirror + checksum-verified fallback), kept on-GPU as one tensor |
| **Teacher** | a hand-built CIFAR ResNet (~1.08M params), trained with plain cross-entropy |
| **Student** | `TinyCNN` (~24k params, **44× smaller**) — the model we'd actually deploy |
| **Loss** | `T`-softened KL divergence (× `T²`), mixed with hard cross-entropy via `ALPHA` |
| **Loop** | SGD + OneCycleLR, GPU-side augmentation, one `train()` for every experiment |
| **Evaluation** | accuracy, t-SNE of features, confusion matrices, attention maps |
| **DKD** *(optional)* | decoupling the soft target into target-class and non-target-class parts |

**What we saw**
- The tiny student gained **~+15–20 points** (≈51% → ~70%) from distillation — *same* network, *same* labels, only
  the teacher's soft targets added — and reached **~78% with no labels at all**.
- **Hard pseudo-labels** help over training alone, but **soft targets win** because they preserve inter-class
  structure (Part 14) — the dark knowledge from Part 7 is real, not just a nicer plot.
- **DKD** (Part 15) often squeezes another **+1–3 points** from the *same* cached teacher logits.
- KD's value is largest when **labels are scarce but unlabelled data and a strong teacher are available**.
- The improvement is visible in the **feature space**, the **confusion matrix**, and in **where the model
  looks** — not just the headline number.
- But the student stays **capacity-limited**: KD *narrows* the teacher–student gap dramatically, it does not
  *close* it (~70% vs ~92%).

**Try it yourself**
1. Sweep `N_LABELS` ∈ {500, 1000, 4000, 50000}: when does KD stop helping, and why?
2. Sweep `TEMP` ∈ {1, 4, 16} and `ALPHA`: what does temperature actually change?
3. Sweep `BETA_DKD` ∈ {1, 4, 8, 16}: how much does the non-target term matter?
4. Compare *transfer-set* KD (`labeled=labeled_idx`) with *soft+hard on the subset only*
   (`train(student, labeled_idx, 60, teacher_logits=TL[labeled_idx])`) — why is using all the images so much better?

**What we left out** (your next topics to explore) — the other rows of the Part 4 table, in code:
- **Feature-based KD** — FitNets (Romero+ 2015), Attention Transfer (Zagoruyko & Komodakis 2017): match intermediate
  features (both models here expose `.fmap(x)` to start from).
- **Relational KD** (Park+ 2019): transfer the structure *between* samples, not single outputs.
- **Online / self-distillation** — Deep Mutual Learning (Zhang+ 2018), Born-Again Networks (Furlanello+ 2018):
  distil without a separate frozen teacher.

### Where to go next

- The [BMVA CVSS Beginner Practical](https://colab.research.google.com/github/Adaptive-Intelligence-Lab/BMVA-summer-school/blob/main/BMVA_CVSS_Beginner.ipynb) — if you haven't done it yet, it covers the CNN and
  training-loop fundamentals this notebook builds on
- [dl-pytorch](https://github.com/atapour/dl-pytorch) — Amir Atapour-Abarghouei's teaching examples (GANs, VAEs, transformers and more)
- [BMVA](https://www.bmva.org/) — the British Machine Vision Association

---
# References

- Hinton, G., Vinyals, O., & Dean, J. (2015). *Distilling the Knowledge in a Neural Network.*
  NeurIPS 2014 Deep Learning Workshop. https://arxiv.org/abs/1503.02531
- Zhao, B., Cui, Q., Song, R., Qiu, Y., & Liang, J. (2022). *Decoupled Knowledge Distillation.* CVPR 2022.
  https://arxiv.org/abs/2203.08679
- Gou, J., Yu, B., Maybank, S. J., & Tao, D. (2021). *Knowledge Distillation: A Survey.* IJCV.
  https://arxiv.org/abs/2006.05525
- Romero, A., et al. (2015). *FitNets: Hints for Thin Deep Nets.* ICLR 2015. https://arxiv.org/abs/1412.6550
- Zagoruyko, S., & Komodakis, N. (2017). *Paying More Attention to Attention.* ICLR 2017.
  https://arxiv.org/abs/1612.03928
- Park, W., Kim, D., Lu, Y., & Cho, M. (2019). *Relational Knowledge Distillation.* CVPR 2019.
  https://arxiv.org/abs/1904.05068
- Krizhevsky, A. (2009). *Learning Multiple Layers of Features from Tiny Images* (CIFAR-10).
  https://www.cs.toronto.edu/~kriz/cifar.html
- `torchvision` documentation: https://pytorch.org/vision/stable/index.html

---

*Created for the BMVA Summer School, as the advanced practical following the
[BMVA CVSS Beginner Practical](https://colab.research.google.com/github/Adaptive-Intelligence-Lab/BMVA-summer-school/blob/main/BMVA_CVSS_Beginner.ipynb). Adapted from the original
knowledge-distillation notebook by [Dr Ruochen Li](http://adaptive-intelligence.co.uk/dr-ruochen-li/), restructured to match the beginner practical's format.*